# KNN

In [6]:
from libs.data_configuration import get_data_partitioned

X_train, X_test, y_train, y_test = get_data_partitioned() # 20% datos para validacion por defecto 

In [23]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, f1_score, recall_score
from sklearn.preprocessing import StandardScaler
import numpy as np

neigborhoods = [3, 5, 7, 11, 13]
weights = ['uniform', 'distance']

rows = []
for n in neigborhoods: 
    for w in weights:
        
        accs_train = []
        accs_val = []
        f1_scores = []
        recall_scores = []
        
        kf = KFold(n_splits=5)
        
        for train_index, test_index in kf.split(X_test): 
            xtrain, xtest = X_test[train_index], X_test[test_index]
            ytrain, ytest = y_test[train_index], y_test[test_index]
            
            scaler = StandardScaler() 
            xtrain, xtest = scaler.fit_transform(xtrain), scaler.transform(xtest)

            model = KNeighborsClassifier(
                p=2, 
                weights=w, 
                n_neighbors=n
            )

            model.fit(xtrain, ytrain)

            predictions_train = model.predict(xtrain)
            predictions_val = model.predict(xtest)
            
            accs_train.append(accuracy_score(ytrain, predictions_train))
            accs_val.append(accuracy_score(ytest, predictions_val))
            f1_scores.append(f1_score(ytest, predictions_val))
            recall_scores.append(recall_score(ytest, predictions_val))
            
        rows.append({
            'numero de vecinos': n,
            'pesos': w,
            'media acc train': np.mean(accs_train), 
            'media acc validacion': np.mean(accs_val),
            'IC acc validacion': np.std(accs_val),
            'media f1': np.mean(f1_scores),
            "IC f1": np.std(f1_scores),
            'media recall': np.mean(recall_scores),
            'IC recall': np.std(recall_scores)
        })

In [24]:
import pandas as pd

results = pd.DataFrame(rows)
results.sort_values(by='media f1', ascending=False)

,numero de vecinos,pesos,media acc train,media acc validacion,IC acc validacion,media f1,IC f1,media recall,IC recall
7,11,distance,1.000000,0.9255,0.007810,0.925635,0.008445,0.931586,0.013168
9,13,distance,1.000000,0.9250,0.005244,0.925347,0.004906,0.932623,0.009921
3,5,distance,1.000000,0.9245,0.006205,0.924769,0.007274,0.932506,0.013445
1,3,distance,1.000000,0.9225,0.007583,0.923096,0.007605,0.933728,0.018250
5,7,distance,1.000000,0.9220,0.011113,0.922701,0.009103,0.931721,0.008712
2,5,uniform,0.941875,0.9225,0.006124,0.922511,0.007634,0.927437,0.014482
6,11,uniform,0.927750,0.9200,0.010954,0.919843,0.012228,0.923447,0.019947
0,3,uniform,0.957375,0.9190,0.008602,0.919268,0.008541,0.925712,0.018319
4,7,uniform,0.935250,0.9180,0.012390,0.918381,0.011254,0.924614,0.014029
8,13,uniform,0.925250,0.9165,0.008888,0.916532,0.008401,0.919571,0.014171


Finalmente se escoge el modelo que mejor puntuacion saco, KNN con pesos basados en la distanca y 11 vecinos

In [22]:
model = KNeighborsClassifier(
    weights='distance', 
    n_neighbors=11
)

model.fit(X_train_nrm, y_train)
predictions = model.predict(X_test_nrm)

accuracy_score(y_test, predictions), f1_score(y_test, predictions), recall_score(y_test, predictions)

(0.951, 0.951048951048951, 0.954864593781344)